## MAGE v.1.1 - Gene expression data processing using DESeq2
---

### *Table of contents*
- [Background](#background)
- [Set-up and data preprocessing](#set-up-and-data-preprocessing)
- [Expression matrices](#expression-matrices)
- [Differential expression analysis](#differential-expression-analysis)
- [Environment](#environment)

### Background

This notebook details the steps taken to process gene expression data (measured using `salmon` and described [here](https://github.com/mccoy-lab/MAGE/blob/2ad80f736d6c6932e746456300cbd82c68092032/analysis_pipeline/01_data_preparation/expression_quantification/README.md)) using the `DESeq2` package in R. The products of this notebook are the following:
- Raw count, estimated mean ($\mu$), and variance-stabilizing transformed (VST) expression matrices.
- Differential expression summary statistics for each continental group label.

The pipeline detailed below follows the best-practices described in the `DESeq2` manual, which can be found [here](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html).

### Set-up and data preprocessing

The data pre-requisites for this pipeline are the following:
1. A metadata file describing the biological (e.g., *continental group* and *sex* labels) and technical (e.g., *batch*) covariates for each sample.
2. A `.bed` file containing a list of genes to be included in the analysis. Here, we use a list of genes which meet experiment-wide expression thresholds (described [here](https://github.com/mccoy-lab/MAGE/blob/2ad80f736d6c6932e746456300cbd82c68092032/analysis_pipeline/01_data_preparation/expression_quantification/README.md)).
3. A directory of `salmon` quantification files, one per sample, containing the estimated counts for each gene.
4. A reference genome annotation file (e.g., `.gtf`) for creating a `tximport` transcript-to-gene mapping object (for gene-level analysis).

First, we will load the library prerequisites.

In [1]:
# Load required libraries
invisible(
  c(library(DESeq2),
    library(tidyverse),
    library(tximport),
    library(tximportData),
    library(txdbmaker),
    library(yaml))
)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The follo

Next, we will load the data prerequisites outlined above. Paths to these files are stored in `03_deseq_config.yaml`.  

#### 1. Metadata

In [25]:
# Load config yaml
deseq_config <- read_yaml("resources/03_deseq_config.yaml")

# Read in metadata and reformat for DESeq2
metadata <- suppressMessages(read_tsv(deseq_config$metadata)) %>%
  as.data.frame()
rownames(metadata) <- metadata$internal_libraryID
metadata$batch <- as.factor(metadata$batch)
metadata$continentalGroup <- as.factor(metadata$continentalGroup)
metadata$population <- as.factor(metadata$population)
metadata$sex <- as.factor(metadata$sex)

# Print number of samples in dataset
cat("Number of samples:", length(unique(metadata$internal_libraryID)), "\n")
cat("Number of unique samples:", length(unique(metadata$sample_kgpID)), "\n")

Number of samples: 779 
Number of unique samples: 731 


From the `metadata` object, we'll extract a dataframe of the replicate samples.

In [3]:
# Identify replicate samples
replicated_samples <- metadata %>%
  dplyr::select(sample_kgpID) %>%
  group_by(sample_kgpID) %>%
  summarise(n_libraries = n()) %>%
  filter(n_libraries > 1) %>%
  pull(sample_kgpID)

# Create a 2-column dataframe (internal_libraryID, sample_kgpID)
replicate_mapping <- metadata %>%
  dplyr::select(internal_libraryID, sample_kgpID) %>%
  filter(sample_kgpID %in% replicated_samples) %>%
  arrange(sample_kgpID)

# Preview replicate mapping
cat("Replicate mapping:")
head(replicate_mapping)

Replicate mapping:

,internal_libraryID,sample_kgpID
,<chr>,<chr>
HG00110-8-1,HG00110-8-1,HG00110
HG00110-8-2,HG00110-8-2,HG00110
HG00110,HG00110,HG00110
HG00121-2-1,HG00121-2-1,HG00121
HG00121-2-2,HG00121-2-2,HG00121
HG00121,HG00121,HG00121


#### 2. Filtered gene list

In [4]:
# Read in filtered expression bed
filtered_genes <- suppressMessages(
  read_tsv(deseq_config$filtered_expression_bed,
           col_select = c(1, 4))
) %>%
  rename(chr = 1, gene = 2) %>%
  filter(!grepl("chrUn|chrM|chrY", chr))

# Print number of genes
cat("Total number of genes:", nrow(filtered_genes), "\n")

Total number of genes: 20154 


#### 3. Salmon quantification files

In [5]:
# Read parent directory containing all salmon quants
salmon_quants_dir <- deseq_config$salmon_dir

quant_files <- list.files(path = salmon_quants_dir,
                          pattern = "quant.sf",
                          recursive = TRUE,
                          full.names = TRUE) %>%
  grep(x = ., pattern = "old", value = TRUE, invert = TRUE)

# Extract library IDs from quant file paths (corresponds to internal_libraryID in metadata)
library_IDs <- sapply(X = quant_files, FUN = function(x) {
  str_split(x, "/") %>%
    unlist() %>%
    tail(n = 2) %>%
    head(n = 1)
})

# Add names to quant files vector
names(quant_files) <- library_IDs

The code block below tests concordance between `quant.sf` sample names and the metadata.

In [6]:
err <- FALSE

## Check that each name of quant_files[i]
## is contained in the value of quant_files[i]
for (i in seq_along(quant_files)) {
  if (!grepl(names(quant_files)[i], quant_files[i])) {
    err <- TRUE
    stop(paste("Name of quant file",
               names(quant_files)[i],
               "is not contained in its path:",
               quant_files[i]))
  }
}

## Check that each name of quant_files is in metadata$internal_libraryID
if (any(!names(quant_files) %in% metadata$internal_libraryID)) {
  err <- TRUE
  stop("Some quant file names are not in metadata$internal_libraryID.")
}

## Report success if no errors were found
if (!err) {
  cat("All quant file names are valid and present in metadata.\n")
}

All quant file names are valid and present in metadata.


#### 4. Tximport transcript-to-gene mapping object

To create a `tximport` transcript-to-gene mapping object, we will use the `GenomicFeatures` package to read in the Gencode annotation file (stored in `deseq_config$gencode`) and create a `TxDb` object.

In [7]:
# Read in Gencode annotation and create TxDb object
txdb <- makeTxDbFromGFF(deseq_config$gencode, format = "gff3")

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
“The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored.”
Warning message in .extract_transcripts_from_GRanges(tx_IDX, gr, mcols0$type, mcols0$ID, :
“the transcript names ("tx_name" column in the TxDb object) imported from the
  "transcript_id" attribute are not unique”
Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
“genome version information is not available for this TxDb object”
OK



In [8]:
# Extract transcript keys
k <- keys(txdb, keytype = "TXNAME")

# Create tx2gene mapping dataframe
tx2gene <- select(txdb, keys = k, "GENEID", "TXNAME")

# Preview tx2gene mapping
head(tx2gene)

'select()' returned 1:many mapping between keys and columns



,TXNAME,GENEID
,<chr>,<chr>
1,ENST00000456328.2,ENSG00000223972.5
2,ENST00000450305.2,ENSG00000223972.5
3,ENST00000473358.1,ENSG00000243485.5
4,ENST00000469289.1,ENSG00000243485.5
5,ENST00000607096.1,ENSG00000284332.1
6,ENST00000606857.1,ENSG00000268020.3


### Expression matrices

With the data pre-requisites loaded above, we will create a `DESeqDataSet` object and use it to create raw count, estimated mean ($\mu$), and variance-stabilizing transformed (VST) expression matrices.

First, we will use `tximport` to read in the `salmon` quantification files and summarize to gene level.

In [ ]:
# Run tximport
txi <- suppressMessages(tximport(files = quant_files,
                type = "salmon",
                tx2gene = tx2gene))

Using the `txi` object created by `tximport`, we will create a `DESeqDataSet` object and collapse technical replicates.

In [27]:
# If txi$counts and metadata are not in the same order,
# reorder metadata to match txi$counts
if (!all(colnames(txi$counts) == rownames(metadata))) {
  metadata <- metadata[colnames(txi$counts), ]
}

# Test that metadata and txi$counts are in the same order
if (all(colnames(txi$counts) == rownames(metadata))) {
  cat("metadata and txi$counts are in the same order.\n")
} else {
  stop("metadata and txi$counts are not in the same order.")
}

metadata and txi$counts are in the same order.


In [ ]:
# Create DESeqDataSet
dds <- DESeqDataSetFromTximport(
  txi = txi,
  colData = metadata,
  design = ~ population + sex + batch
)

# Collapse technical replicates
dds <- collapseReplicates(object = dds,
                          groupby = dds$sample_kgpID,
                          run = dds$internal_libraryID)

# Run DESeq2 pipeline
## Specify multithreading parameters
BiocParallel::register(BiocParallel::MulticoreParam(workers = 16))

## Run DESeq2
dds <- DESeq(dds, parallel = TRUE)

using counts and average transcript lengths from tximport

Warning message in collapseReplicates(object = dds, groupby = dds$sample_kgpID, :
“

  Warning! collapseReplicates only sums columns of the 'counts' assay.
  Other assays are dropped from output, and must be manually combined,
  as it is not unambiguous how to combine non-count assays.
”
estimating size factors

estimating dispersions

gene-wise dispersion estimates: 16 workers



Note that the warning produced above is irrelevant for our purposes, as we are only operating at the scale of counts.

#### Generate expression matrices

In [45]:
# Generate count matrix
count_matrix <- counts(dds)

# Generate VST matrix
vst_matrix <- vst(dds, blind = TRUE)

# Glimpse of matrices
count_matrix[1:5, 1:5]
assay(vst_matrix)[1:5, 1:5]

# Write to files
write_csv(x = as.data.frame(count_matrix),
          file = "results/expression.counts.csv",
          col_names = TRUE)
write_csv(x = as.data.frame(assay(vst_matrix)),
          file = "results/expression.vst.csv",
          col_names = TRUE)

,HG00096,HG00100,HG00105,HG00108,HG00110
ENSG00000000003.15,2,1,3,0,23
ENSG00000000005.6,0,0,0,0,0
ENSG00000000419.14,959,948,1394,1199,4127
ENSG00000000457.14,208,168,234,173,715
ENSG00000000460.17,207,184,192,198,656


,HG00096,HG00100,HG00105,HG00108,HG00110
ENSG00000000003.15,5.186844,5.049784,5.229257,4.718241,5.514619
ENSG00000000005.6,4.718241,4.718241,4.718241,4.718241,4.718241
ENSG00000000419.14,10.446592,10.426072,10.646486,10.600739,10.577744
ENSG00000000457.14,8.417801,8.153743,8.279475,8.058404,8.257966
ENSG00000000460.17,8.411844,8.263535,8.042210,8.219855,8.154062


In [47]:
vst(counts(dds), blind = TRUE)[1:5, 1:5]

,HG00096,HG00100,HG00105,HG00108,HG00110
ENSG00000000003.15,5.186844,5.049784,5.229257,4.718241,5.514619
ENSG00000000005.6,4.718241,4.718241,4.718241,4.718241,4.718241
ENSG00000000419.14,10.446592,10.426072,10.646486,10.600739,10.577744
ENSG00000000457.14,8.417801,8.153743,8.279475,8.058404,8.257966
ENSG00000000460.17,8.411844,8.263535,8.042210,8.219855,8.154062
